# Phase 1: Autoregressive models.

•Implement and train a simple MLP language model akin to Figure 1 in [2] that takes in the
three previous tokens (amino acids) and predicts the next, using a deep learning framework
such as PyTorch. The model architecture and hyperparameters are up to you, but clearly state
and justify your choices in the report.


•Implement and train a recurrent neural network (RNN) model or a causal decoder-only multi-
head attention Transformer model [3]. For the RNN choose between the vanilla RNN, Gated
Recurrent Unit (GRU) or Long Short-Term Memory (LSTM). Like the MLP, the specific model
design is up to you but needs to be explained and justified in the report.


•Evaluate and visualize the performance of these models by folding the generated sequences
with ESMFold [4] and presenting relevant metrics (code for evaluation will be provided).

---

## Importing the data

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#device = torch.device(
#    "mps" if torch.backends.mps.is_available() else "cpu"
#)
#print("Using:", device)

In [3]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

## Defining the transformer

### Iniatiing embedings

For the embeddings we are intializing a trainable lookup table 

Num_embeddings is the vocab size 

Embeding dim, is an arbitary number we define, we define it as 128. But if our model is to weak, we can go back and increase it. 

It works by applying the embeding to each element in an input 

\<BOS>, A, C, \<EOS>, \<PAD>

x = torch.tensor([[1, 3, 4, 2, 0]])  # Shape: [1, 5]

embedded_x = token_embedding(x)     # Shape: [1, 5, 128]

In [4]:
token_embedding = nn.Embedding(
    num_embeddings=train_data.vocab_size,
    embedding_dim=128,
    padding_idx=train_data.vocab["<PAD>"], # PyTorch initializes that embedding row to zeros and excludes it from embedding-gradient updates
)

We now add positional embedings and combine them with our token embeding 

In [ ]:
position_embedding = nn.Embedding(
    num_embeddings=train_data.max_length,
    embedding_dim=128,
)

# Run it on one batch to check the shapes
x, targets, family, lengths = next(iter(train_loader))


positions = torch.arange(x.size(1), device=x.device)
h = token_embedding(x) + position_embedding(positions)

print("Input shape:", x.shape)       # [32, 129]
print("Embedding shape:", h.shape)   # [32, 129, 128]

Input shape: torch.Size([32, 129])
Embedding shape: torch.Size([32, 129, 128])


### Building the casual mask 

In [ ]:
T = x.size(1)
causal_mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool, device=x.device),
    diagonal=1,
)